In [44]:
2+2

4

In [45]:
import torch
import torch.nn as nn
import math
import numpy as np

In [46]:

class Tokenizer():
    def __init__(self):
        pass

    def tokenize(self, text):
        text = text.lower()
        text = text.replace('?', '')
        text = text.replace("'", '')
        return text.split()    



In [47]:

class PositionalEncoding(nn.Module):
    def __init__(self, embedding_dim=512, max_seq_len=5000):
        super().__init__()

        # Create matrix
        pe = torch.zeros(max_seq_len, embedding_dim)

        # Position: 0, 1, 2, 3, ...
        position = torch.arange(0,max_seq_len,dtype=torch.float).unsqueeze(1)

        # Division term
        div_term = torch.exp(torch.arange(0,embedding_dim,2).float()* (-math.log(10000.0) / embedding_dim))

        # Even dimensions -> sin
        pe[:, 0::2] = torch.sin(position * div_term)

        # Odd dimensions -> cos
        pe[:, 1::2] = torch.cos(position * div_term)

        # Add batch dimension
        pe = pe.unsqueeze(0)

        # Register as buffer
        self.register_buffer('pe', pe)


    def forward(self, x):

        # x shape:
        # (sequence_length, embedding_dim)

        seq_len = x.size(0)

        positional_encoding = self.pe[0,:seq_len,:]

        return positional_encoding

In [48]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim=512, num_heads=16):
        super().__init__()

        self.num_heads = num_heads 
        self.embedding_dim = embedding_dim 

        # dimension of each head
        self.head_dim = embedding_dim // num_heads 

        # Q, K, V
        self.W_Q = nn.Linear(embedding_dim, embedding_dim)
        self.W_K = nn.Linear(embedding_dim, embedding_dim)
        self.W_V = nn.Linear(embedding_dim, embedding_dim)

        # final linear layer
        self.W_O = nn.Linear(embedding_dim, embedding_dim)


    def forward(self, input_features):  


        Q = self.W_Q(input_features)  
        K = self.W_K(input_features)  
        V = self.W_V(input_features) 

        seq_len = input_features.shape[0] 

        Q = Q.reshape(seq_len,self.num_heads,self.head_dim) 
        K = K.reshape(seq_len,self.num_heads,self.head_dim) 
        V = V.reshape(seq_len,self.num_heads,self.head_dim) 

        heads = []
        for head in range(self.num_heads):
            q = Q[:, head, :]
            k = K[:, head, :]
            v = V[:, head, :]

            # QK^T
            scores = q @ k.T

            # scale
            scores = scores / math.sqrt(self.head_dim)

            # softmax
            weights = torch.softmax(scores, dim=-1)

            # weighted sum of V
            output = weights @ v

            heads.append(output)

        multi_head = torch.cat(heads, dim=-1) 
        output = self.W_O(multi_head) 
        return output








In [49]:

class MaskedMultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim=512, num_heads=16):
        super().__init__()

        self.num_heads = num_heads
        self.embedding_dim = embedding_dim

        # dimension of each head
        self.head_dim = embedding_dim // num_heads

        # Q, K, V
        self.W_Q = nn.Linear(embedding_dim, embedding_dim)
        self.W_K = nn.Linear(embedding_dim, embedding_dim)
        self.W_V = nn.Linear(embedding_dim, embedding_dim)

        # final linear layer
        self.W_O = nn.Linear(embedding_dim, embedding_dim)


    def forward(self, input_features):

        Q = self.W_Q(input_features)
        K = self.W_K(input_features)
        V = self.W_V(input_features)

        seq_len = input_features.shape[0]

        Q = Q.reshape(seq_len, self.num_heads, self.head_dim)
        K = K.reshape(seq_len, self.num_heads, self.head_dim)
        V = V.reshape(seq_len, self.num_heads, self.head_dim)

        heads = []

        for head in range(self.num_heads):

            q = Q[:, head, :]
            k = K[:, head, :]
            v = V[:, head, :]

            # QK^T
            scores = q @ k.T

            # scale
            scores = scores / math.sqrt(self.head_dim)

            # =====================
            # Causal Mask
            # =====================

            mask = torch.triu(
                torch.ones(seq_len, seq_len, device=input_features.device),
                diagonal=1
            )

            scores = scores.masked_fill(mask == 1, float('-inf'))

            # softmax
            weights = torch.softmax(scores, dim=-1)

            # weighted sum of V
            output = weights @ v

            heads.append(output)

        multi_head = torch.cat(heads, dim=-1)

        output = self.W_O(multi_head)

        return output



In [50]:
class FeedForward(nn.Module):
    def __init__(self, embedding_dim=512, ff_dim=2048):
        super().__init__()

        # First projection: 512 -> 2048
        self.linear1 = nn.Linear(embedding_dim, ff_dim)

        # ReLU activation
        self.relu = nn.ReLU()

        # Second projection: 2048 -> 512
        self.linear2 = nn.Linear(ff_dim, embedding_dim)

    def forward(self, x):
        # x shape: (sequence_length, 512)

        x = self.linear1(x)   # (seq_len, 2048)
        x = self.relu(x)      # (seq_len, 2048)
        x = self.linear2(x)   # (seq_len, 512)

        return x



In [51]:
class WordEmbedding(nn.Module):

    def __init__(self, vocab_size, embedding_dim=512):
        super().__init__()

        self.embedding = nn.Embedding(num_embeddings=vocab_size,embedding_dim=embedding_dim)

    def forward(self, token_ids):
        embeddings = self.embedding(token_ids)
        return embeddings



In [52]:
class CrossMultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim=512, num_heads=16):
        super().__init__()

        self.num_heads = num_heads
        self.embedding_dim = embedding_dim

        # dimension of each head
        self.head_dim = embedding_dim // num_heads

        # Query comes from decoder
        self.W_Q = nn.Linear(embedding_dim, embedding_dim)

        # Key and Value come from encoder
        self.W_K = nn.Linear(embedding_dim, embedding_dim)
        self.W_V = nn.Linear(embedding_dim, embedding_dim)

        # final linear layer
        self.W_O = nn.Linear(embedding_dim, embedding_dim)


    def forward(self, decoder_features, encoder_output):

        # Query from decoder
        Q = self.W_Q(decoder_features)

        # Key and Value from encoder
        K = self.W_K(encoder_output)
        V = self.W_V(encoder_output)

        decoder_seq_len = decoder_features.shape[0]
        encoder_seq_len = encoder_output.shape[0]

        Q = Q.reshape(
            decoder_seq_len,
            self.num_heads,
            self.head_dim
        )

        K = K.reshape(
            encoder_seq_len,
            self.num_heads,
            self.head_dim
        )

        V = V.reshape(
            encoder_seq_len,
            self.num_heads,
            self.head_dim
        )

        heads = []

        for head in range(self.num_heads):

            q = Q[:, head, :]
            k = K[:, head, :]
            v = V[:, head, :]

            # QK^T
            scores = q @ k.T

            # scale
            scores = scores / math.sqrt(self.head_dim)

            # softmax
            weights = torch.softmax(scores, dim=-1)

            # weighted sum of V
            output = weights @ v

            heads.append(output)

        multi_head = torch.cat(heads, dim=-1)

        output = self.W_O(multi_head)

        return output



In [53]:
class AddAndNormalize(nn.Module):
    def __init__(self, embedding_dim=512):
        super().__init__()

        self.layer_norm = nn.LayerNorm(embedding_dim)

    def forward(self, x, sublayer_output):
        # Add residual connection
        x = x + sublayer_output

        # Normalize
        x = self.layer_norm(x)

        return x



In [54]:

class Encoder(nn.Module):

    def __init__(
        self,
        num_layers=6,
        embedding_dim=512,
        num_heads=16,
        ff_dim=2048
    ):
        super().__init__()

        self.positional_encoding = PositionalEncoding(
            embedding_dim=embedding_dim
        )

        self.multi_head_attention = MultiHeadAttention(
            embedding_dim=embedding_dim,
            num_heads=num_heads
        )

        self.add_norm = AddAndNormalize(
            embedding_dim=embedding_dim
        )

        self.feed_forward = FeedForward(
            embedding_dim=embedding_dim,
            ff_dim=ff_dim
        )

        self.num_layers = num_layers
        self.embedding_dim = embedding_dim

    def forward(self, embeddings):

        positional_encoding = self.positional_encoding(
            embeddings
        )

        x = embeddings + positional_encoding

        for _ in range(self.num_layers):

            attention_output = self.multi_head_attention(
                x
            )

            attention_output = self.add_norm(
                x,
                attention_output
            )

            feed_forward_output = self.feed_forward(
                attention_output
            )

            x = self.add_norm(
                attention_output,
                feed_forward_output
            )

        return x

In [55]:
class Decoder(nn.Module):

    def __init__(
        self,
        num_layers=16,
        embedding_dim=512,
        num_heads=16,
        ff_dim=2048
    ):
        super().__init__()

        self.positional_encoding = PositionalEncoding(
            embedding_dim=embedding_dim
        )

        self.masked_multi_head_attention = MaskedMultiHeadAttention(
            embedding_dim=embedding_dim,
            num_heads=num_heads
        )

        self.cross_multi_head_attention = CrossMultiHeadAttention(
            embedding_dim=embedding_dim,
            num_heads=num_heads
        )

        self.add_norm = AddAndNormalize(
            embedding_dim=embedding_dim
        )

        self.feed_forward = FeedForward(
            embedding_dim=embedding_dim,
            ff_dim=ff_dim
        )

        self.num_layers = num_layers
        self.embedding_dim = embedding_dim

    def forward(self, embeddings, encoder_output):

        positional_encoding = self.positional_encoding(
            embeddings
        )

        x = embeddings + positional_encoding

        for _ in range(self.num_layers):

            # Masked self-attention

            masked_attention_output = self.masked_multi_head_attention(
                x
            )

            masked_attention_output = self.add_norm(
                x,
                masked_attention_output
            )

            # Cross-attention

            cross_attention_output = self.cross_multi_head_attention(
                masked_attention_output,
                encoder_output
            )

            cross_attention_output = self.add_norm(
                masked_attention_output,
                cross_attention_output
            )

            # Feed-forward network

            feed_forward_output = self.feed_forward(
                cross_attention_output
            )

            x = self.add_norm(
                cross_attention_output,
                feed_forward_output
            )

        return x

In [ ]:

# =========================================================
# DEVICE
# =========================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# =========================================================
# TRAINING DATA
# =========================================================

# CHANGE THIS PATH
# Example:
# /kaggle/input/shakespeare/input.txt

input_path = "/kaggle/input/datasets/prabhavsinghal/text-data/input.txt"

with open(
    input_path,
    "r",
    encoding="utf-8"
) as file:

    text = file.read()


texts = [
    line.strip()
    for line in text.splitlines()
    if line.strip()
]


print("Number of training lines:", len(texts))


# =========================================================
# TOKENIZER
# =========================================================

tokenizer = Tokenizer()


# =========================================================
# BUILD VOCABULARY
# =========================================================

vocab = {
    "<PAD>": 0,
    "<UNK>": 1,
    "<BOS>": 2,
    "<EOS>": 3
}


for text in texts:

    tokens = tokenizer.tokenize(text)

    for token in tokens:

        if token not in vocab:

            vocab[token] = len(vocab)


id_to_token = {
    index: token
    for token, index in vocab.items()
}


vocab_size = len(vocab)


print("Vocabulary size:", vocab_size)


# =========================================================
# MODEL SETTINGS
# =========================================================

embedding_dim = 64
num_heads = 4
ff_dim = 128
num_layers = 2


# =========================================================
# ENCODER
# =========================================================

encoder = Encoder(
    num_layers=num_layers,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    ff_dim=ff_dim
).to(device)


# =========================================================
# DECODER
# =========================================================

decoder = Decoder(
    num_layers=num_layers,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    ff_dim=ff_dim
).to(device)


# =========================================================
# EMBEDDING LAYERS
# =========================================================

encoder_embedding = WordEmbedding(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim
).to(device)


decoder_embedding = WordEmbedding(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim
).to(device)


# =========================================================
# OUTPUT LAYER
# =========================================================

output_layer = nn.Linear(
    embedding_dim,
    vocab_size
).to(device)


# =========================================================
# LOSS
# =========================================================

criterion = nn.CrossEntropyLoss()


# =========================================================
# OPTIMIZER
# =========================================================

optimizer = torch.optim.Adam(
    list(encoder.parameters())
    + list(decoder.parameters())
    + list(encoder_embedding.parameters())
    + list(decoder_embedding.parameters())
    + list(output_layer.parameters()),
    lr=0.001
)


# =========================================================
# TRAINING SETTINGS
# =========================================================

epochs = 1000


# =========================================================
# TRAINING
# =========================================================

for epoch in range(epochs):

    total_loss = 0.0


    for text_index, text in enumerate(texts):

        # =================================================
        # TOKENIZE
        # =================================================

        tokens = tokenizer.tokenize(text)


        # Skip empty token sequences

        if len(tokens) == 0:
            continue


        # =================================================
        # TOKEN IDS
        # =================================================

        token_ids = torch.tensor(
            [
                vocab.get(
                    token,
                    vocab["<UNK>"]
                )
                for token in tokens
            ],
            dtype=torch.long,
            device=device
        )


        # =================================================
        # TARGET
        # =================================================

        # Example:
        #
        # the cat sits
        #
        # becomes:
        #
        # the cat sits <EOS>

        target_ids = torch.cat(
            [
                token_ids,
                torch.tensor(
                    [vocab["<EOS>"]],
                    dtype=torch.long,
                    device=device
                )
            ]
        )


        # =================================================
        # ENCODER
        # =================================================

        encoder_embeddings = encoder_embedding(
            token_ids
        )

        encoder_output = encoder(
            encoder_embeddings
        )


        # =================================================
        # RIGHT SHIFT
        # =================================================

        # Target:
        #
        # the cat sits <EOS>
        #
        # Decoder input:
        #
        # <BOS> the cat sits

        decoder_input_ids = torch.cat(
            [
                torch.tensor(
                    [vocab["<BOS>"]],
                    dtype=torch.long,
                    device=device
                ),
                target_ids[:-1]
            ]
        )


        # =================================================
        # DECODER EMBEDDINGS
        # =================================================

        decoder_embeddings = decoder_embedding(
            decoder_input_ids
        )


        # =================================================
        # DECODER
        # =================================================

        decoder_output = decoder(
            decoder_embeddings,
            encoder_output
        )


        # =================================================
        # VOCABULARY LOGITS
        # =================================================

        logits = output_layer(
            decoder_output
        )


        # =================================================
        # LOSS
        # =================================================

        loss = criterion(
            logits,
            target_ids
        )


        # =================================================
        # BACKPROPAGATION
        # =================================================

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()


        total_loss += loss.item()


    # =====================================================
    # AVERAGE LOSS
    # =====================================================

    average_loss = (
        total_loss / len(texts)
    )


    # =====================================================
    # PRINT PROGRESS
    # =====================================================

    if (epoch + 1) % 10 == 0:

        print(
            f"Epoch {epoch + 1}/{epochs} "
            f"Loss: {average_loss:.4f}"
        )


# =========================================================
# SAVE MODEL
# =========================================================

checkpoint = {

    # Vocabulary
    "vocab": vocab,

    # Model weights
    "encoder": encoder.state_dict(),

    "decoder": decoder.state_dict(),

    "encoder_embedding":
        encoder_embedding.state_dict(),

    "decoder_embedding":
        decoder_embedding.state_dict(),

    "output_layer":
        output_layer.state_dict(),

    # Model configuration
    "embedding_dim": embedding_dim,

    "num_heads": num_heads,

    "ff_dim": ff_dim,

    "num_layers": num_layers
}


# =========================================================
# SAVE TO KAGGLE WORKING DIRECTORY
# =========================================================

model_path = "/kaggle/working/model.pth"


torch.save(
    checkpoint,
    model_path
)


print()
print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)

print("Model saved to:")
print(model_path)

print(
    "Model size:",
    f"{os.path.getsize(model_path) / (1024 ** 2):.2f} MB"
)

Device: cuda
GPU: Tesla T4
Number of training lines: 32777
Vocabulary size: 22211


In [ ]:
import torch

from tokenizer import Tokenizer
from embeddings import WordEmbedding

from encoder import Encoder
from decoder import Decoder


# =========================================================
# CONFIGURATION
# =========================================================

CHECKPOINT_PATH = "model.pth"

embedding_dim = 64
num_heads = 4
ff_dim = 128
num_layers = 2


# =========================================================
# TOKENIZER
# =========================================================

tokenizer = Tokenizer()


# =========================================================
# LOAD CHECKPOINT
# =========================================================

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu"
)


# =========================================================
# VOCABULARY
# =========================================================

vocab = checkpoint["vocab"]

id_to_token = {
    index: token
    for token, index in vocab.items()
}

vocab_size = len(vocab)


# =========================================================
# MODEL
# =========================================================

encoder = Encoder(
    num_layers=num_layers,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    ff_dim=ff_dim
)

decoder = Decoder(
    num_layers=num_layers,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    ff_dim=ff_dim
)


# =========================================================
# EMBEDDINGS
# =========================================================

encoder_embedding = WordEmbedding(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim
)

decoder_embedding = WordEmbedding(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim
)


# =========================================================
# LOAD WEIGHTS
# =========================================================

encoder.load_state_dict(
    checkpoint["encoder"]
)

decoder.load_state_dict(
    checkpoint["decoder"]
)

encoder_embedding.load_state_dict(
    checkpoint["encoder_embedding"]
)

decoder_embedding.load_state_dict(
    checkpoint["decoder_embedding"]
)


# =========================================================
# OUTPUT LAYER
# =========================================================

import torch.nn as nn

output_layer = nn.Linear(
    embedding_dim,
    vocab_size
)

output_layer.load_state_dict(
    checkpoint["output_layer"]
)


# =========================================================
# EVALUATION MODE
# =========================================================

encoder.eval()
decoder.eval()
encoder_embedding.eval()
decoder_embedding.eval()
output_layer.eval()


# =========================================================
# INFERENCE
# =========================================================

def generate(
    text,
    max_length=10
):

    # =====================================================
    # TOKENIZE INPUT
    # =====================================================

    tokens = tokenizer.tokenize(text)

    if len(tokens) == 0:
        return ""


    # =====================================================
    # CONVERT INPUT TO IDS
    # =====================================================

    token_ids = torch.tensor(
        [
            vocab.get(
                token,
                vocab["<UNK>"]
            )
            for token in tokens
        ],
        dtype=torch.long
    )


    # =====================================================
    # ENCODER
    # =====================================================

    with torch.no_grad():

        encoder_embeddings = encoder_embedding(
            token_ids
        )

        encoder_output = encoder(
            encoder_embeddings
        )


        # =================================================
        # START DECODER WITH <BOS>
        # =================================================

        decoder_input_ids = torch.tensor(
            [vocab["<BOS>"]],
            dtype=torch.long
        )


        generated_tokens = []


        # =================================================
        # AUTOREGRESSIVE GENERATION
        # =================================================

        for _ in range(max_length):

            decoder_embeddings = decoder_embedding(
                decoder_input_ids
            )

            decoder_output = decoder(
                decoder_embeddings,
                encoder_output
            )

            logits = output_layer(
                decoder_output
            )


            # =============================================
            # LAST POSITION
            # =============================================

            last_logits = logits[-1]


            # =============================================
            # GET MOST LIKELY TOKEN
            # =============================================

            next_token_id = torch.argmax(
                last_logits
            ).item()


            # =============================================
            # EOS
            # =============================================

            if next_token_id == vocab["<EOS>"]:
                break


            # =============================================
            # IGNORE SPECIAL TOKENS
            # =============================================

            if next_token_id not in [
                vocab["<PAD>"],
                vocab["<BOS>"]
            ]:

                next_token = id_to_token[
                    next_token_id
                ]

                generated_tokens.append(
                    next_token
                )


            # =============================================
            # APPEND TOKEN TO DECODER INPUT
            # =============================================

            next_token_tensor = torch.tensor(
                [next_token_id],
                dtype=torch.long
            )

            decoder_input_ids = torch.cat(
                [
                    decoder_input_ids,
                    next_token_tensor
                ]
            )


    # =====================================================
    # RETURN GENERATED TEXT
    # =====================================================

    return " ".join(
        generated_tokens
    )


# =========================================================
# INTERACTIVE INFERENCE
# =========================================================

if __name__ == "__main__":

    print("Model loaded.")
    print("Type 'quit' to exit.")
    print()

    while True:

        text = input(
            "Input: "
        ).strip()


        if text.lower() == "quit":
            break


        result = generate(
            text,
            max_length=10
        )


        print(
            "Generated:",
            result
        )